<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.7_Praktikum_Training_Neural_Network.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.7: Praktikum Optimasi Penurunan Gradien Lanjut
## Mata Kuliah: Kecerdasan Buatan dan Sains Data Pertanian Presisi (INSTIPER)

---

### Tujuan Praktikum:
1. Mengimplementasikan 4 keluarga algoritma optimizer murni dari dasar (*from scratch*): **SGD**, **Momentum**, **RMSprop**, dan **Adam** menggunakan NumPy.
2. Memvisualisasikan dan membandingkan trajektori konvergensi masing-masing optimizer pada lanskap jurang sempit (*ravine surface*).
3. Mengimplementasikan teknik penjadwalan laju pembelajaran canggih: **Cosine Annealing Schedule**.
4. Melatih model regresi Multi-Layer Perceptron untuk peramalan defisit air tanah multi-blok perkebunan kelapa sawit.
5. Membandingkan kecepatan konvergensi, kestabilan rugi, dan akurasi $R^2$ antar-optimizer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Pengaturan visualisasi publikasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka untuk eksplorasi Advanced Optimizers berhasil dimuat!")

## 1. Implementasi 4 Optimizer Murni Berbasis Kelas Modular (Pure NumPy)
Di bawah ini kita membangun kelas `OptimizerSGD`, `OptimizerMomentum`, `OptimizerRMSprop`, dan `OptimizerAdam` lengkap dengan pembaruan momen dan koreksi bias analitik.

In [ ]:
class OptimizerSGD:
    def __init__(self, lr=0.01):
        self.lr = lr
    def update(self, params, grads):
        for k in params:
            params[k] -= self.lr * grads[f'd{k}']

class OptimizerMomentum:
    def __init__(self, lr=0.01, gamma=0.9):
        self.lr = lr
        self.gamma = gamma
        self.v = {}
    def update(self, params, grads):
        for k in params:
            if k not in self.v:
                self.v[k] = np.zeros_like(params[k])
            self.v[k] = self.gamma * self.v[k] + self.lr * grads[f'd{k}']
            params[k] -= self.v[k]

class OptimizerRMSprop:
    def __init__(self, lr=0.001, beta=0.9, eps=1e-8):
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.s = {}
    def update(self, params, grads):
        for k in params:
            if k not in self.s:
                self.s[k] = np.zeros_like(params[k])
            g = grads[f'd{k}']
            self.s[k] = self.beta * self.s[k] + (1.0 - self.beta) * (g**2)
            params[k] -= (self.lr / (np.sqrt(self.s[k]) + self.eps)) * g

class OptimizerAdam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = {}
        self.v = {}
        self.t = 0
    def update(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.m:
                self.m[k] = np.zeros_like(params[k])
                self.v[k] = np.zeros_like(params[k])
            g = grads[f'd{k}']
            # Momen 1 (kecepatan) dan Momen 2 (skala)
            self.m[k] = self.beta1 * self.m[k] + (1.0 - self.beta1) * g
            self.v[k] = self.beta2 * self.v[k] + (1.0 - self.beta2) * (g**2)
            # Koreksi bias analitik
            m_hat = self.m[k] / (1.0 - self.beta1**self.t)
            v_hat = self.v[k] / (1.0 - self.beta2**self.t)
            # Pembaruan parameter terbobot
            params[k] -= (self.lr / (np.sqrt(v_hat) + self.eps)) * m_hat

print("[OK] Seluruh 4 kelas Optimizer berhasil didefinisikan secara sempurna!")

## 2. Simulasi 2D: Perbandingan Trajektori pada Lanskap Jurang Curam
Kita menguji keempat optimizer pada fungsi kuadratik berjurang sempit $f(x, y) = 0.05 x^2 + 1.2 y^2$ dengan titik awal $(-5.0, 2.5)$ untuk mengamati bagaimana masing-masing algoritma menangani osilasi sumbu vertikal.

In [ ]:
# Parameter fungsi objektif kuadratik: f(w1, w2) = 0.05*w1^2 + 1.2*w2^2
start_pos = np.array([-5.0, 2.5])
n_steps = 30

def run_simulation(opt_class, **opt_kwargs):
    opt = opt_class(**opt_kwargs)
    params = {'w': start_pos.copy()}
    path = [params['w'].copy()]
    for _ in range(n_steps):
        # Gradien analitik: df/dw1 = 0.1*w1, df/dw2 = 2.4*w2
        g = np.array([0.1 * params['w'][0], 2.4 * params['w'][1]])
        grads = {'dw': g}
        opt.update(params, grads)
        path.append(params['w'].copy())
    return np.array(path)

path_sgd = run_simulation(OptimizerSGD, lr=0.35)
path_mom = run_simulation(OptimizerMomentum, lr=0.25, gamma=0.8)
path_rms = run_simulation(OptimizerRMSprop, lr=0.45, beta=0.9)
path_adam = run_simulation(OptimizerAdam, lr=0.65, beta1=0.9, beta2=0.999)

# Visualisasi Kontur
x_mesh = np.linspace(-6.0, 6.0, 300)
y_mesh = np.linspace(-3.5, 3.5, 300)
Xm, Ym = np.meshgrid(x_mesh, y_mesh)
Zm = 0.05 * Xm**2 + 1.2 * Ym**2

plt.figure(figsize=(10, 6))
cs = plt.contour(Xm, Ym, Zm, levels=[0.2, 0.8, 2.0, 5.0, 10.0, 18.0], colors='#bdc3c7')
plt.clabel(cs, inline=True, fontsize=8)

plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='#e74c3c', marker='o', markersize=3.5, label='SGD (Osilasi Curam)', lw=1.5)
plt.plot(path_mom[:, 0], path_mom[:, 1], color='#f39c12', marker='s', markersize=3.5, label='Momentum (Inersia)', lw=1.8)
plt.plot(path_rms[:, 0], path_rms[:, 1], color='#2980b9', marker='^', markersize=4, label='RMSprop (Adaptif)', lw=1.8)
plt.plot(path_adam[:, 0], path_adam[:, 1], color='#27ae60', marker='D', markersize=4.5, label='Adam (Cepat & Stabil)', lw=2.2)

plt.scatter([0], [0], color='#8e44ad', s=160, marker='*', edgecolors='black', label='Target Minimum Global (0, 0)', zorder=10)
plt.title('Simulasi Trajektori 4 Optimizer pada Lanskap Jurang Sempit', fontweight='bold', pad=12)
plt.xlabel('Parameter Sumbu Datar (w1)')
plt.ylabel('Parameter Sumbu Curam (w2)')
plt.legend(loc='lower right', framealpha=0.9)
plt.xlim(-6.0, 6.0)
plt.ylim(-3.5, 3.5)
plt.show()

## 3. Dataset Agro-Industri: Peramalan Defisit Air Tanah Multi-Blok
Kita membangkitkan 800 sampel data mikroklimat perkebunan sawit untuk memprediksi defisit air tanah kumulatif (mm).
- Fitur: Suhu Udara ($^\circ\text{C}$), Kelembaban Relatif (%), Radiasi Matahari ($\text{MJ/m}^2$), Kecepatan Angin (m/s), dan Hari Tanpa Hujan.

In [ ]:
np.random.seed(42)
n_samples = 800

suhu = np.random.uniform(24.0, 36.0, n_samples)
kelembaban = np.random.uniform(50.0, 95.0, n_samples)
radiasi = np.random.uniform(10.0, 26.0, n_samples)
angin = np.random.uniform(0.5, 4.0, n_samples)
hari_kering = np.random.uniform(0.0, 22.0, n_samples)

# Dinamika biologis defisit air tanah (non-linier)
defisit_air = (
    12.0 +
    2.8 * (suhu - 24.0) -
    0.8 * (kelembaban - 50.0) +
    2.2 * (radiasi - 10.0) +
    4.5 * (angin - 0.5) +
    6.8 * hari_kering +
    0.15 * (hari_kering ** 1.8) +
    np.random.normal(0, 4.0, n_samples)
)
defisit_air = np.clip(defisit_air, 0.0, 250.0)

X_data = np.column_stack([suhu, kelembaban, radiasi, angin, hari_kering])
y_data = defisit_air.reshape(-1, 1)

# Split & Scale
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.25, random_state=42)

scaler_x = StandardScaler()
X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print(f"Dataset Agro-Klimat: {len(X_train_scaled)} sampel latih, {len(X_test_scaled)} sampel uji.")

## 4. Implementasi MLP Regresi dengan Antarmuka Optimizer Modular
Kita menyusun arsitektur MLP `5 -> 32 -> 16 -> 1` dengan fungsi rugi MSE yang dapat dipasangkan dengan sembarang optimizer modular.

In [ ]:
class MLPRegressorMultiOpt:
    def __init__(self, layer_dims=[5, 32, 16, 1], seed=42):
        self.layer_dims = layer_dims
        self.L = len(layer_dims) - 1
        self.params = {}
        rng = np.random.RandomState(seed)
        for l in range(1, self.L + 1):
            n_in, n_out = layer_dims[l-1], layer_dims[l]
            self.params[f'W{l}'] = rng.normal(0, np.sqrt(2.0 / n_in), size=(n_in, n_out))
            self.params[f'b{l}'] = np.zeros((1, n_out))

    def relu(self, Z):
        return np.maximum(0.0, Z)

    def relu_deriv(self, Z):
        return np.where(Z > 0.0, 1.0, 0.0)

    def forward(self, X):
        cache = {'A0': X}
        A = X
        # Hidden layers (ReLU)
        for l in range(1, self.L):
            Z = np.dot(A, self.params[f'W{l}']) + self.params[f'b{l}']
            A = self.relu(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A
        # Output layer (Linear)
        Z_last = np.dot(A, self.params[f'W{self.L}']) + self.params[f'b{self.L}']
        cache[f'Z{self.L}'] = Z_last
        cache[f'A{self.L}'] = Z_last
        return Z_last, cache

    def backward(self, cache, Y):
        grads = {}
        B = Y.shape[0]
        # MSE Output Delta: (y_hat - y) / B
        delta = (cache[f'A{self.L}'] - Y) / B
        grads[f'dW{self.L}'] = np.dot(cache[f'A{self.L-1}'].T, delta)
        grads[f'db{self.L}'] = np.sum(delta, axis=0, keepdims=True)
        
        for l in range(self.L - 1, 0, -1):
            delta = np.dot(delta, self.params[f'W{l+1}'].T) * self.relu_deriv(cache[f'Z{l}'])
            grads[f'dW{l}'] = np.dot(cache[f'A{l-1}'].T, delta)
            grads[f'db{l}'] = np.sum(delta, axis=0, keepdims=True)
        return grads

    def train_with_optimizer(self, X, Y, optimizer, epochs=60, batch_size=32):
        loss_hist = []
        n_samples = X.shape[0]
        for ep in range(epochs):
            perm = np.random.permutation(n_samples)
            X_s = X[perm]
            Y_s = Y[perm]
            for b in range(0, n_samples, batch_size):
                xb = X_s[b:b+batch_size]
                yb = Y_s[b:b+batch_size]
                preds, cache = self.forward(xb)
                grads = self.backward(cache, yb)
                optimizer.update(self.params, grads)
                
            all_preds, _ = self.forward(X)
            mse = 0.5 * np.mean((all_preds - Y)**2)
            loss_hist.append(mse)
        return loss_hist

print("[OK] Kelas MLPRegressorMultiOpt berhasil disiapkan!")

## 5. Eksperimen Komparasi Pelatihan: SGD vs Momentum vs RMSprop vs Adam
Kita melatih model dengan inisialisasi bobot identik untuk menguji performa keempat optimizer selama 60 epoch.

In [ ]:
epochs_count = 60
opt_configs = {
    'SGD': OptimizerSGD(lr=0.04),
    'Momentum': OptimizerMomentum(lr=0.03, gamma=0.85),
    'RMSprop': OptimizerRMSprop(lr=0.008, beta=0.9),
    'Adam': OptimizerAdam(lr=0.015, beta1=0.9, beta2=0.999)
}

histories = {}
trained_models = {}

for name, opt in opt_configs.items():
    # Inisialisasi model dengan seed identik agar titik awal sama persis
    model = MLPRegressorMultiOpt(layer_dims=[5, 32, 16, 1], seed=100)
    loss_history = model.train_with_optimizer(X_train_scaled, y_train_scaled, opt, epochs=epochs_count, batch_size=32)
    histories[name] = loss_history
    trained_models[name] = model
    print(f"[{name}] Selesai! Loss Akhir: {loss_history[-1]:.4f}")

## 6. Visualisasi Kurva Konvergensi & Evaluasi Metrik Bisnis
Kita memetakan kurva penurunan loss seluruh optimizer dan menghitung skor determinasi ($R^2$) pada data uji independen.

In [ ]:
plt.figure(figsize=(9, 5))
colors = {'SGD': '#e74c3c', 'Momentum': '#f39c12', 'RMSprop': '#2980b9', 'Adam': '#27ae60'}

for name, hist in histories.items():
    plt.plot(range(1, epochs_count + 1), hist, label=name, color=colors[name], lw=2.2)

plt.title('Kurva Penurunan Kerugian MSE pada Pelatihan Model Defisit Air Sawit', fontweight='bold', pad=12)
plt.xlabel('Epoch Pelatihan')
plt.ylabel('Training Loss (MSE Ternormalisasi)')
plt.legend(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Evaluasi pada Data Uji Asli (Inverse Transform)
summary_data = []
for name, model in trained_models.items():
    preds_scaled, _ = model.forward(X_test_scaled)
    preds_real = scaler_y.inverse_transform(preds_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, preds_real))
    r2 = r2_score(y_test, preds_real)
    summary_data.append({
        'Optimizer': name,
        'Loss Akhir': round(histories[name][-1], 4),
        'RMSE Data Uji (mm)': round(rmse, 2),
        'Skor R2': round(r2, 4),
        'Efisiensi Konvergensi': 'Sangat Cepat' if name in ['Adam', 'RMSprop'] else 'Sedang / Berosilasi'
    })

df_summary = pd.DataFrame(summary_data)
print("=== TABEL KOMPARASI PERFORMA PREDIKSI DEFISIT AIR TANAH ===")
print(df_summary.to_string(index=False))